In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings; warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from keras.models import Sequential
from keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

n_in=7
n_out=2
n_dim = 100
use_sw = False
with open('stopwords_english') as f:
    stopwords = set(f.read().splitlines())

df = pd.read_csv('test.csv', header=None, names=['label','title','content']).dropna(subset=['content'])
def longest_sentence(text):
    parts = [s.strip() for s in re.split(r'[.!?]', str(text)) if s.strip()]
    return max(parts, key=lambda s: len(s.split())) if parts else str(text)

def clean(text, drop_sw=True):
    s = longest_sentence(text).lower()
    s = re.sub(r'\d+', '', s)
    s = re.sub(r'[^\w\s]', '', s)
    toks = [t for t in s.split() if t.isalpha()]
    if drop_sw:
        toks = [t for t in toks if t not in stopwords]
    return toks

sents = [clean(t, drop_sw=not use_sw) for t in df['content']]
sents = [s for s in sents if len(s) >= n_in + n_out]

In [ ]:
def split_sequences(seq, n_in, n_out):
    X, y = [], []
    for i in range(len(seq)):
        end, out_end = i + n_in, i + n_in + n_out
        if out_end > len(seq): break
        X.append(seq[i:end, :])
        y.append(seq[end:out_end, :])
    return np.array(X), np.array(y)

def build_xy(sents, w2v):
    Xs, ys = [], []
    for s in sents:
        vecs = [w2v.wv[w] for w in s if w in w2v.wv]
        if len(vecs) < n_in + n_out: continue
        seq = np.vstack(vecs)
        X, y = split_sequences(seq, n_in, n_out)
        Xs.append(X); ys.append(y)
    return np.vstack(Xs), np.vstack(ys)

def make_lstm(act):
    m = Sequential()
    m.add(LSTM(200, activation=act, input_shape=(n_in, n_dim)))
    m.add(RepeatVector(n_out))
    m.add(LSTM(200, activation=act, return_sequences=True))
    m.add(TimeDistributed(Dense(n_dim)))
    m.compile(optimizer='adam', loss='mse')
    return m

w2v = Word2Vec(sentences=sents, vector_size=n_dim, window=5, min_count=1, sg=1, epochs=10, seed=42)
X, y = build_xy(sents, w2v)

m_tanh = make_lstm('tanh'); m_tanh.fit(X, y, epochs=20, batch_size=256, verbose=0)
m_relu = make_lstm('relu'); m_relu.fit(X, y, epochs=20, batch_size=256, verbose=0)

err_tanh = np.sqrt(np.mean((m_tanh.predict(X, verbose=0) - y)**2, axis=(1,2)))
err_relu = np.sqrt(np.mean((m_relu.predict(X, verbose=0) - y)**2, axis=(1,2)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(err_tanh, bins=50, color='steelblue'); ax[0].set_title(f'tanh ({err_tanh.mean():.3f})'); ax[0].set_xlabel('RMSE')
ax[1].hist(err_relu, bins=50, color='coral'); ax[1].set_title(f'relu ({err_relu.mean():.3f})'); ax[1].set_xlabel('RMSE')
plt.tight_layout(); plt.show()

In [ ]:
if err_tanh.mean() <= err_relu.mean():
    best_act, best_model, best_err = 'tanh', m_tanh, err_tanh
else:
    best_act, best_model, best_err = 'relu', m_relu, err_relu

lengths = [len(s) for s in sents]
i_long = int(np.argmax(lengths))
doc = sents[i_long]

vecs = np.vstack([w2v.wv[w] for w in doc if w in w2v.wv])
x_in = vecs[-(n_in+n_out):-n_out].reshape(1, n_in, n_dim)
y_true = vecs[-n_out:]
yhat = best_model.predict(x_in, verbose=0)

print('\nclosest words to predicted vector 1:')
for w, sim in w2v.wv.similar_by_vector(yhat[0, 0, :], topn=5):
    print(f'  {w}  ({sim:.3f})')

print('\nclosest words to predicted vector 2:')
for w, sim in w2v.wv.similar_by_vector(yhat[0, 1, :], topn=5):
    print(f'  {w}  ({sim:.3f})')

In [ ]:
sents_sw = [clean(t, drop_sw=False) for t in df['content']]
sents_sw = [s for s in sents_sw if len(s) >= n_in + n_out]
w2v_sw = Word2Vec(sentences=sents_sw, vector_size=n_dim, window=5, min_count=1, sg=1, epochs=10, seed=42)
X_sw, y_sw = build_xy(sents_sw, w2v_sw)
m_sw = make_lstm(best_act); m_sw.fit(X_sw, y_sw, epochs=20, batch_size=256, verbose=0)
err_sw = np.sqrt(np.mean((m_sw.predict(X_sw, verbose=0) - y_sw)**2, axis=(1,2)))

print(f'no stopwords RMSE: {best_err.mean():.4f}')
print(f'with stopwords RMSE: {err_sw.mean():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(best_err, bins=50, color='steelblue'); ax[0].set_title(f'no sw ({best_err.mean():.3f})'); ax[0].set_xlabel('RMSE')
ax[1].hist(err_sw, bins=50, color='darkorange'); ax[1].set_title(f'with sw ({err_sw.mean():.3f})'); ax[1].set_xlabel('RMSE')
plt.tight_layout(); plt.show()

print(f'no stopwords RMSE: {best_err.mean():.4f}')
print(f'with stopwords RMSE: {err_sw.mean():.4f}')
use_sw = False


From the results, we can see that removing stop words gave the lower RMSE. This is because when we allow the stop words to stay in the input window the input window will contain words like 'the', which will decrease the actual content of that the LSTM gets to try to learn from. The closest word match is also more on topic when there are no stop words